<a href="https://colab.research.google.com/github/vectara/example-notebooks/blob/main/notebooks/api-examples/11-web-get-tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vectara `web_get`: Calling REST APIs from an Agent

This notebook shows how to give a Vectara agent the built-in **`web_get`** tool so it can issue HTTP requests to any REST API at conversation time.

`web_get` is a general-purpose HTTP client for the agent. It supports:

- Any HTTP method: `GET`, `POST`, `PUT`, `DELETE`, `HEAD`
- Arbitrary request `headers` (including `Authorization` for private/authenticated APIs)
- A request `body` (JSON, form data, or anything else your API accepts)

That makes it suitable for both **retrieval** (read data — public APIs, internal services, partner endpoints) and **actions** (create a ticket, post a message, update a record, kick off a workflow). Anywhere your existing systems already speak HTTP, an agent can drive them with `web_get`.

Unlike `corpora_search` (which queries indexed content) or `web_search` (which goes through a search engine), `web_get` lets the agent talk to a **specific endpoint** you've described.

For clarity, the demo in this notebook calls a **public, no-auth REST API** ([Open-Meteo](https://open-meteo.com/)) so it runs out of the box for any reader. The same patterns apply unchanged to authenticated APIs and to non-GET methods — see the notes in Step 4.

You'll learn how to:
1. Configure an agent with the inline `web_get` tool
2. Have the agent call a real REST API end-to-end
3. Inspect the `tool_input` / `tool_output` events to see exactly which request the agent made and what it got back
4. Constrain the tool with `argument_override` to pin auth headers, method, timeout, and response size in production

## Why `web_get`?

`web_get` is the agent's general-purpose HTTP client — for anything that isn't already in a Vectara corpus and isn't a web-search-style question:

| Tool | What it does | When to use it |
|---|---|---|
| `corpora_search` | Hybrid search over indexed corpora | Your data lives in Vectara |
| `web_search` | Search-engine-style web search | Open-domain questions |
| `web_get` | Issue an HTTP request (`GET`/`POST`/`PUT`/`DELETE`/`HEAD`) to a specific endpoint, with custom headers and body | Calling a specific REST API — public or authenticated, read or write |
| `lambda` | Run sandboxed Python | Custom computation, no network access |

Some things `web_get` enables, beyond the public-data demo in this notebook:

- **Private / authenticated APIs** — pin an `Authorization: Bearer <token>` header (or any other auth scheme) via `argument_override.headers` and the agent can call your internal services or partner APIs.
- **State-changing actions** — the LLM (or your `argument_override`) can set `method="POST"`/`"PUT"`/`"DELETE"` and a request `body` to file a ticket, send a message, update a record, or trigger a workflow.
- **Multi-step API choreography** — the agent can chain calls, feeding fields from one response into the next request (the Open-Meteo demo below does exactly this in two steps).

Under the hood the LLM picks `url`, `method`, `headers`, `body`, etc. on each turn — or you can pin any of those with `argument_override` (Step 4).

## Getting Started

All you need for this notebook is a `VECTARA_API_KEY`. 

The REST API the demo calls (Open-Meteo) is a free, no-signup, no-auth public endpoint — chosen so this notebook runs unmodified for everyone. To point `web_get` at a private API instead, you'd pin an `Authorization` header via `argument_override.headers` (see Step 4); nothing else in the notebook changes.

This notebook is self-contained — it does not depend on the corpora, agents, or tools created in earlier notebooks.

## Setup

In [1]:
import os
import json
from datetime import datetime

import requests

api_key = os.environ['VECTARA_API_KEY']

BASE_URL = "https://api.vectara.io/v2"

headers = {
    "x-api-key": api_key,
    "Content-Type": "application/json"
}

In [2]:
# Load the shared helpers (delete_and_create_agent).
# vectara_utils.py sits next to this notebook in the repo; Colab fetches it on
# first run so the same notebook works locally and in a fresh Colab runtime.
try:
    from vectara_utils import delete_and_create_agent
except ImportError:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/vectara/example-notebooks/main/notebooks/api-examples/vectara_utils.py",
        "vectara_utils.py",
    )
    from vectara_utils import delete_and_create_agent

import vectara_utils
vectara_utils.configure(BASE_URL, headers)

## Step 1: Define an Agent with the `web_get` Tool

We'll build a **Weather Assistant** that answers natural-language weather questions by calling [Open-Meteo](https://open-meteo.com/), a free public weather API.

Open-Meteo needs two calls per question, which makes for a rich `web_get` demo:

1. **Geocode** the city: `GET https://geocoding-api.open-meteo.com/v1/search?name=<city>&count=1` → returns latitude/longitude
2. **Forecast**: `GET https://api.open-meteo.com/v1/forecast?latitude=<lat>&longitude=<lon>&current=temperature_2m,weather_code,wind_speed_10m` → returns current conditions

The agent's instructions describe the two-step flow; the LLM then chooses the URLs and parameters at conversation time. The tool itself is configured inline as `{"type": "web_get"}` — no `argument_override` yet, so the LLM is free to pick everything.

In [3]:
WEATHER_AGENT_NAME = "Weather Assistant"

weather_instructions = """You are a helpful weather assistant. You answer questions about current weather conditions by calling the free Open-Meteo public API using the `web_get` tool.

For each user question, you MUST follow this exact two-step flow:

Step 1 - Geocode the city to get latitude and longitude:
  GET https://geocoding-api.open-meteo.com/v1/search?name=<CITY>&count=1&language=en&format=json
  Read `results[0].latitude` and `results[0].longitude` from the JSON response.

Step 2 - Fetch current weather using those coordinates:
  GET https://api.open-meteo.com/v1/forecast?latitude=<LAT>&longitude=<LON>&current=temperature_2m,weather_code,wind_speed_10m&temperature_unit=celsius&wind_speed_unit=kmh
  Read `current.temperature_2m`, `current.weather_code`, and `current.wind_speed_10m` from the JSON response.

Always use HTTP method GET. Never invent values - if a call fails, say so honestly.

Reply to the user with a short, friendly two-sentence summary of the current weather (temperature in Celsius, wind speed in km/h, and a plain-English description of the weather code). Do not show raw JSON."""

weather_agent_config = {
    "name": WEATHER_AGENT_NAME,
    "description": "Answers weather questions by calling the Open-Meteo public API via the web_get tool.",
    "model": {"name": "gpt-4o"},
    "first_step_name": "main",
    "steps": {
        "main": {
            "instructions": [
                {
                    "type": "inline",
                    "name": "weather_instructions",
                    "template": weather_instructions,
                }
            ],
            "output_parser": {"type": "default"},
        }
    },
    "tool_configurations": {
        "web_get": {
            "type": "web_get"
        }
    },
}

weather_agent_key = delete_and_create_agent(weather_agent_config, WEATHER_AGENT_NAME)

Created agent 'Weather Assistant' (key: agt_weather_assistant_c73f)


## Step 2: Create a Session

Sessions hold the conversation state. Each user question becomes one event posted to the session.

In [4]:
session_name = f"Weather Demo {datetime.now().strftime('%Y%m%d-%H%M%S')}"
session_config = {"name": session_name, "metadata": {"purpose": "web_get_demo"}}

response = requests.post(
    f"{BASE_URL}/agents/{weather_agent_key}/sessions",
    headers=headers,
    json=session_config,
)
response.raise_for_status()
session_key = response.json()["key"]
print(f"Session Created: {session_key}")

Session Created: ase_weather_demo_20260506-100555_ec6c


## Step 3: Ask a Question that Requires Live Data

We'll send a question and inspect the events the agent emits. The interesting events are:

- `tool_input` for `web_get` → the URL/method/headers the LLM chose
- `tool_output` for `web_get` → the HTTP status and (a snippet of) the response body
- `agent_output` → the natural-language answer to the user

The helper below pulls those out of the response so we can see the agent's reasoning trace, not just the final answer.

In [5]:
def ask_weather(agent_key, session_key, question, show_events=True, body_preview_chars=300, return_events=False):
    """Send a question to the agent and (optionally) print every tool call it makes.

    Prints each tool_input/tool_output event prefixed with the tool name
    (`tool_configuration_name`), so the trace works whether the agent is using
    a single generic `web_get` tool (Steps 3 and 4) or several specialized ones
    like `geocode_city` / `get_current_weather` (Step 5).

    By default returns just the agent's final reply (a string). Pass
    ``return_events=True`` to get back ``(answer, events)`` so you can pass the
    raw events list to ``find_tool_failures`` (introduced in Step 7).
    """
    payload = {
        "messages": [{"type": "text", "content": question}],
        "stream_response": False,
    }
    url = f"{BASE_URL}/agents/{agent_key}/sessions/{session_key}/events"
    response = requests.post(url, headers=headers, json=payload)
    if response.status_code != 201:
        print(f"Error: {response.status_code} - {response.text}")
        return (None, []) if return_events else None

    events = response.json().get("events", [])
    if show_events:
        print("\n------ Agent Events ------")
        for event in events:
            etype = event.get("type", "unknown")
            tool_name = event.get("tool_configuration_name")
            if etype == "tool_input" and tool_name:
                ti = event.get("tool_input", {})
                method = ti.get("method", "GET")
                target_url = ti.get("url", "<no url>")
                print(f"[{tool_name}] {method} {target_url}")
                if ti.get("headers"):
                    print(f"  headers: {ti['headers']}")
                if ti.get("body"):
                    body_preview = ti["body"]
                    if isinstance(body_preview, (dict, list)):
                        body_preview = json.dumps(body_preview)
                    body_preview = str(body_preview)
                    if len(body_preview) > body_preview_chars:
                        body_preview = body_preview[:body_preview_chars] + "..."
                    print(f"  body: {body_preview}")
            elif etype == "tool_output" and tool_name:
                to = event.get("tool_output", {})
                # tool_output uses `status_code` and `content` (verified against the live API).
                status = to.get("status_code", "?")
                body = to.get("content", "")
                if isinstance(body, (dict, list)):
                    body = json.dumps(body)
                body = str(body)
                if len(body) > body_preview_chars:
                    body = body[:body_preview_chars] + "..."
                print(f"  -> [{tool_name}] HTTP {status}, body: {body}")
        print("-" * 26)

    answer = next(
        (e.get("content") for e in events if e.get("type") == "agent_output"),
        None,
    )
    return (answer, events) if return_events else answer

In [6]:
question = "What's the weather in Tokyo right now?"
print(f"User: {question}")
print("=" * 80)

answer = ask_weather(weather_agent_key, session_key, question, show_events=True)
print(f"\nAgent: {answer}")

User: What's the weather in Tokyo right now?

------ Agent Events ------
[web_get] GET https://geocoding-api.open-meteo.com/v1/search?name=Tokyo&count=1&language=en&format=json
  -> [web_get] HTTP 200, body: {"results":[{"id":1850147,"name":"Tokyo","latitude":35.6895,"longitude":139.69171,"elevation":44.0,"feature_code":"PPLC","country_code":"JP","admin1_id":1850144,"timezone":"Asia/Tokyo","population":9733276,"country_id":1861060,"country":"Japan","admin1":"Tokyo"}],"generationtime_ms":0.6380081}
[web_get] GET https://api.open-meteo.com/v1/forecast?latitude=35.6895&longitude=139.69171&current=temperature_2m,weather_code,wind_speed_10m&temperature_unit=celsius&wind_speed_unit=kmh
  -> [web_get] HTTP 200, body: {"latitude":35.7,"longitude":139.6875,"generationtime_ms":0.10943412780761719,"utc_offset_seconds":0,"timezone":"GMT","timezone_abbreviation":"GMT","elevation":40.0,"current_units":{"time":"iso8601","interval":"seconds","temperature_2m":"°C","weather_code":"wmo code","wind_speed_

Notice the trace: the agent issued **two** `web_get` calls in sequence — first to the geocoding endpoint, then to the forecast endpoint, using the latitude/longitude it parsed from the first response. The LLM picked every part of those URLs (host, path, query string) at runtime, guided only by the system prompt.

## Step 4: Constrain the Tool with `argument_override`

In production you usually don't want to leave every HTTP knob to the LLM. The inline `web_get` tool accepts an **`argument_override`** block that pins specific parameters; anything you set there is fixed and the LLM cannot change it. Anything you leave out, the LLM still fills in per call.

Common things to pin:

- `headers`: pin an `Authorization: Bearer <token>` (or `X-API-Key: ...`, basic auth, etc.) when calling a private/authenticated API; pin a `User-Agent` for attribution; pin `Content-Type: application/json` for write endpoints. **Heads-up: `argument_override.headers` *replaces* the headers the LLM tries to set — it is not a merge.** Pin every header you care about in one dict, or leave `headers` unset entirely if you want the LLM to add per-call headers (e.g. a notification `Title`/`Priority`).
- `method`: pin `"GET"` if you want a read-only agent that physically cannot `POST`/`PUT`/`DELETE`. Conversely, if your agent is supposed to *act* (e.g. always file a ticket), you can pin `"POST"`.
- `url`: pin a specific endpoint when the agent should only ever talk to one resource. Leave it unset (as we do below) if the LLM needs to choose between several endpoints per turn.
- `body`: pin a fixed request body (e.g. a constant template) when only the URL or query parameters should vary.
- `timeout_seconds`: prevent slow APIs from blowing your turn budget.
- `max_content_bytes`: cap response size so a giant payload doesn't blow the LLM context.
- `follow_redirects`, `ssl_verify`: enforce safe defaults.

Below we lock down headers, method, timeout, and limits — but deliberately leave `url` unset, because the demo agent still needs to choose between the geocoding endpoint and the forecast endpoint per turn. For an authenticated API, the same pattern applies: just add an `Authorization` header to `argument_override.headers` (alongside any other headers you want, since the dict replaces wholesale).

In [7]:
weather_agent_config_locked = {
    **weather_agent_config,
    "description": "Locked-down weather agent: web_get headers, method, and limits are pinned via argument_override.",
    "tool_configurations": {
        "web_get": {
            "type": "web_get",
            "argument_override": {
                "method": "GET",
                "headers": {"User-Agent": "vectara-tutorial/1.0"},
                "follow_redirects": True,
                "timeout_seconds": 10,
                "max_content_bytes": 50000,
                "ssl_verify": True,
            },
        }
    },
}

weather_agent_key = delete_and_create_agent(weather_agent_config_locked, WEATHER_AGENT_NAME)

Deleted existing agent 'Weather Assistant' (agt_weather_assistant_c73f)
Created agent 'Weather Assistant' (key: agt_weather_assistant_c409)


In [8]:
# Fresh session against the recreated agent.
session_name = f"Weather Demo Locked {datetime.now().strftime('%Y%m%d-%H%M%S')}"
response = requests.post(
    f"{BASE_URL}/agents/{weather_agent_key}/sessions",
    headers=headers,
    json={"name": session_name, "metadata": {"purpose": "web_get_demo_locked"}},
)
response.raise_for_status()
session_key = response.json()["key"]
print(f"Session Created: {session_key}")

question = "What's the weather in Reykjavik right now?"
print(f"\nUser: {question}")
print("=" * 80)

answer = ask_weather(weather_agent_key, session_key, question, show_events=True)
print(f"\nAgent: {answer}")

Session Created: ase_weather_demo_locked_20260506-100608_9f2e

User: What's the weather in Reykjavik right now?

------ Agent Events ------
[web_get] GET https://geocoding-api.open-meteo.com/v1/search?name=Reykjavik&count=1&language=en&format=json
  headers: {'User-Agent': 'vectara-tutorial/1.0'}
  -> [web_get] HTTP 200, body: {"results":[{"id":3413829,"name":"Reykjavik","latitude":64.13548,"longitude":-21.89541,"elevation":37.0,"feature_code":"PPLC","country_code":"IS","admin1_id":3426182,"admin2_id":3413831,"timezone":"Atlantic/Reykjavik","population":118918,"postcodes":["101","103","104","105","107","108","109","110","...
[web_get] GET https://api.open-meteo.com/v1/forecast?latitude=64.13548&longitude=-21.89541&current=temperature_2m,weather_code,wind_speed_10m&temperature_unit=celsius&wind_speed_unit=kmh
  headers: {'User-Agent': 'vectara-tutorial/1.0'}
  -> [web_get] HTTP 200, body: {"latitude":64.12922,"longitude":-21.883698,"generationtime_ms":0.11265277862548828,"utc_offset_sec

Look at the `headers:` line printed for each `web_get` call: it should now show our pinned `User-Agent: vectara-tutorial/1.0`, regardless of what the LLM might otherwise have chosen. Same for `method` (always `GET`), `timeout_seconds`, and `max_content_bytes` — those are all set by Vectara before the request goes out.

## Step 5: Specialized `web_get` Tools, One per Operation

So far the agent has had a *single* `web_get` registration, with all endpoint guidance baked into the system prompt. That works for a tutorial — but for anything beyond a small surface, the recommended pattern is to register `web_get` **multiple times under different names**, one per logical operation, each with its own `description_template` and `argument_override`.

Why this is better at scale:

- **Sharper tool selection.** LLMs reliably pick the right tool from a short, named list. They drift far more when reading prose in a long system prompt that lists multiple endpoints.
- **Per-operation guardrails.** Each registration carries its own `argument_override`. Pin an `Authorization: Bearer <ticketing_token>` only on the `create_ticket` tool; pin a different timeout for a slow internal API; pin `method=POST` on a write tool so the LLM physically cannot use it for `GET`.
- **Cleaner telemetry.** `tool_input.tool_configuration_name` surfaces as the named operation (e.g. `geocode_city`) rather than a generic `web_get`, which makes traces and audit logs queryable per capability.
- **Method enforcement.** Read-only tools can pin `method="GET"`; action tools can pin `method="POST"`/`"PUT"`/`"DELETE"`.

### But won't the LLM call each tool with arguments?

Yes — and this is the important nuance. The arguments the LLM passes to a `web_get`-typed tool are exactly the fields of `WebGetToolParameters`: `url`, `method`, `headers`, `body`, `follow_redirects`, `timeout_seconds`, `head_lines`, `tail_lines`, `ssl_verify`, `max_content_bytes`. There is **no mechanism to expose domain-typed parameters** like `city: str` or `lat: float` directly. The LLM still constructs the full URL string per call.

So what does the multi-tool pattern give you? **Better tool *selection* + per-tool guardrails — not typed function arguments.**

### One spec constraint to know

`argument_override.url` is a single static string (or `EagerReference`); there's no URL-template substitution. So when an endpoint has variable parts (e.g. a city name in the query string), you typically pin everything **except** `url` and use `description_template` to tell the LLM exactly what URL pattern to construct. That's what we do below.

In [9]:
# Same agent name as Steps 1 and 4 — delete_and_create_agent will replace whichever
# version of the agent currently exists. The system prompt is now much shorter
# because each tool's description_template carries its own URL guidance.

weather_instructions_specialized = """You are a helpful weather assistant. To answer a question about current weather conditions in a city:

1. Call the `geocode_city` tool to resolve the city name to latitude and longitude.
2. Call the `get_current_weather` tool with those coordinates to fetch current conditions.

Reply with a short, friendly two-sentence summary (temperature in Celsius, wind speed in km/h, plain-English description of the weather code). Do not show raw JSON. If a tool call fails, say so honestly and do not invent values."""

_LOCKDOWN = {
    "method": "GET",
    "headers": {"User-Agent": "vectara-tutorial/1.0"},
    "follow_redirects": True,
    "timeout_seconds": 10,
    "max_content_bytes": 50000,
    "ssl_verify": True,
}

weather_agent_config_specialized = {
    "name": WEATHER_AGENT_NAME,
    "description": "Specialized weather agent: separate web_get tools for geocoding and forecasting.",
    "model": {"name": "gpt-4o"},
    "first_step_name": "main",
    "steps": {
        "main": {
            "instructions": [
                {
                    "type": "inline",
                    "name": "weather_instructions_specialized",
                    "template": weather_instructions_specialized,
                }
            ],
            "output_parser": {"type": "default"},
        }
    },
    "tool_configurations": {
        "geocode_city": {
            "type": "web_get",
            "description_template": (
                "Resolve a city name to latitude and longitude using Open-Meteo's "
                "free geocoding API. Construct the URL as: "
                "https://geocoding-api.open-meteo.com/v1/search"
                "?name=<CITY>&count=1&language=en&format=json. "
                "From the JSON response, read results[0].latitude and "
                "results[0].longitude."
            ),
            "argument_override": _LOCKDOWN,
        },
        "get_current_weather": {
            "type": "web_get",
            "description_template": (
                "Fetch current weather at a coordinate using Open-Meteo's free "
                "forecast API. Construct the URL as: "
                "https://api.open-meteo.com/v1/forecast"
                "?latitude=<LAT>&longitude=<LON>"
                "&current=temperature_2m,weather_code,wind_speed_10m"
                "&temperature_unit=celsius&wind_speed_unit=kmh. "
                "From the JSON response, read current.temperature_2m, "
                "current.weather_code, and current.wind_speed_10m."
            ),
            "argument_override": _LOCKDOWN,
        },
    },
}

weather_agent_key = delete_and_create_agent(weather_agent_config_specialized, WEATHER_AGENT_NAME)

Deleted existing agent 'Weather Assistant' (agt_weather_assistant_c409)
Created agent 'Weather Assistant' (key: agt_weather_assistant_7065)


In [10]:
session_name = f"Weather Demo Specialized {datetime.now().strftime('%Y%m%d-%H%M%S')}"
response = requests.post(
    f"{BASE_URL}/agents/{weather_agent_key}/sessions",
    headers=headers,
    json={"name": session_name, "metadata": {"purpose": "web_get_demo_specialized"}},
)
response.raise_for_status()
session_key = response.json()["key"]
print(f"Session Created: {session_key}")

question = "What's the weather in Paris right now?"
print(f"\nUser: {question}")
print("=" * 80)

answer = ask_weather(weather_agent_key, session_key, question, show_events=True)
print(f"\nAgent: {answer}")

Session Created: ase_weather_demo_specialized_20260506-100624_6f5c

User: What's the weather in Paris right now?

------ Agent Events ------
[geocode_city] GET https://geocoding-api.open-meteo.com/v1/search?name=Paris&count=1&language=en&format=json
  headers: {'User-Agent': 'vectara-tutorial/1.0'}
  -> [geocode_city] HTTP 200, body: {"results":[{"id":2988507,"name":"Paris","latitude":48.85341,"longitude":2.3488,"elevation":42.0,"feature_code":"PPLC","country_code":"FR","admin1_id":3012874,"admin2_id":2968815,"admin3_id":2988506,"admin4_id":6455259,"timezone":"Europe/Paris","population":2138551,"postcodes":["75001","75020","7500...
[get_current_weather] GET https://api.open-meteo.com/v1/forecast?latitude=48.85341&longitude=2.3488&current=temperature_2m,weather_code,wind_speed_10m&temperature_unit=celsius&wind_speed_unit=kmh
  headers: {'User-Agent': 'vectara-tutorial/1.0'}
  -> [get_current_weather] HTTP 200, body: {"latitude":48.86,"longitude":2.3399997,"generationtime_ms":0.169157981

Look at the trace: each call's bracketed prefix is now the **operation name** (`geocode_city`, then `get_current_weather`) rather than a generic `web_get`. The LLM picked one of two named capabilities per call, guided by the per-tool `description_template` rather than by prose buried in the system prompt. Every call also inherited the locked `User-Agent`, `method=GET`, and size/timeout limits from each tool's `argument_override`.

### When to use which pattern

| Pattern | Use when |
|---|---|
| **Single `web_get` + prompt guidance** (Steps 1–4) | Tutorials, prototypes, single-endpoint agents, or one-off scripts. |
| **Multiple specialized `web_get` tools** (this Step) | Production agents with several endpoints; you want per-op auth/limits, sharper tool selection, and clean per-capability telemetry. |
| **`lambda` tool** | You need typed function arguments and the work is pure computation (no network). |
| **`InlineMcpToolConfiguration` (MCP)** | You need typed function arguments *and* network access. Stand up a small MCP server that wraps your REST API as proper typed functions. |

## Step 6: `POST` — Sending a Real Action via `web_get`

Up to here every `web_get` call has been a `GET` — the agent has only *read* data. But `web_get` supports any HTTP method, and the introduction promised state-changing actions: filing tickets, sending messages, triggering workflows. Time to actually demonstrate one.

We'll extend the Weather Assistant with a third tool that POSTs to **[ntfy.sh](https://ntfy.sh)**, a free public push-notification service. POSTing a message to `https://ntfy.sh/<topic>` delivers a real notification to anyone subscribed to that topic — there's no signup or auth, just pick a topic name. So this isn't a synthetic echo; it's a genuine outbound action with a side effect we can verify.

> **Public-by-default warning.** Topics on ntfy.sh are public — anyone who knows the topic name can read its messages. We generate a random per-run topic (`vectara-demo-<uuid>`) so this notebook doesn't leak anything sensitive. For real notifications you'd use a private/authenticated push service.

For an authenticated POST API (Slack webhook, your internal ticketing system, etc.) the only thing that changes is adding `Authorization: Bearer <token>` to `argument_override.headers` — same pattern as Step 4 showed for GETs.

In [11]:
import uuid

ALERT_TOPIC = f"vectara-demo-{uuid.uuid4().hex[:8]}"
ALERT_URL = f"https://ntfy.sh/{ALERT_TOPIC}"
print(f"Posting alerts to: {ALERT_URL}")
print("(public topic — anyone with this URL can read its messages)")

Posting alerts to: https://ntfy.sh/vectara-demo-5cbed713
(public topic — anyone with this URL can read its messages)


In [12]:
# Add a third specialized web_get tool: send_weather_alert (POST to ntfy.sh).
# Two key differences from the GET tools above:
#   1. argument_override pins method="POST" AND a fixed url — there's exactly
#      one endpoint we want this tool to talk to.
#   2. We DELIBERATELY do not pin `headers` here. argument_override.headers is
#      replace-mode (see Step 4); pinning even a single User-Agent would silently
#      strip the Title/Priority/Tags the LLM is supposed to choose per call.
#      The LLM still chooses the request body and those required headers per turn.

_POST_LOCKDOWN = {
    "method": "POST",
    "url": ALERT_URL,
    "follow_redirects": True,
    "timeout_seconds": 10,
    "max_content_bytes": 10000,
    "ssl_verify": True,
}

# Define the alert tool as its own top-level dict so cell 25's tool_configurations
# block lines up — all three entries become `"name": <single reference>`,
# matching how `geocode_city` and `get_current_weather` are pulled in.
send_weather_alert_tool = {
    "type": "web_get",
    "description_template": (
        "Send a weather alert as a push notification via ntfy.sh. "
        "Always POST. The URL is pinned by the platform — do not change it. "
        "Set the request `body` to a single short sentence of plain text "
        "(not JSON) summarizing the conditions for a human reader. "
        "You MUST set all three request headers below on every call (do not omit any): "
        "`Title` (short summary), "
        "`Priority` ('1' through '5'; '4' for hot/cold extremes, '3' otherwise), "
        "`Tags` (comma-separated emoji shortcodes like 'snowflake,cold' or 'sun')."
    ),
    "argument_override": _POST_LOCKDOWN,
}

weather_instructions_with_alert = """You are a helpful weather assistant. To answer a question about current weather conditions in a city:

1. Call the `geocode_city` tool to resolve the city name to latitude and longitude.
2. Call the `get_current_weather` tool with those coordinates to fetch current conditions.
3. Call the `send_weather_alert` tool to push a one-sentence summary as a notification. You MUST set ALL THREE of the following request headers on this call (do not omit any):
   - `Title` header: a short summary like "Cold weather alert" or "Pleasant weather in <city>".
   - `Priority` header: "4" if temperature is below 5°C or above 30°C, otherwise "3".
   - `Tags` header: comma-separated emoji shortcodes that match conditions (e.g. "snowflake,cold", "sun", "umbrella,rain").
   And set the request body to a single human-readable sentence (plain text, not JSON) describing the conditions.

Reply to the user with a short, friendly two-sentence summary (temperature in Celsius, wind speed in km/h, plain-English description of the weather code) and mention that an alert was sent. Do not show raw JSON. If a tool call fails, say so honestly and do not invent values."""

# All three tools below are inline `web_get` registrations under custom names;
# the GET tools were defined in Step 5's config (cell 20), the alert tool is
# defined just above. Pulling each into a single reference keeps tool_configurations
# readable.
weather_agent_config_with_alert = {
    "name": WEATHER_AGENT_NAME,
    "description": "Weather agent with a POST tool: fetches conditions and pushes an alert via ntfy.sh.",
    "model": {"name": "gpt-4o"},
    "first_step_name": "main",
    "steps": {
        "main": {
            "instructions": [
                {
                    "type": "inline",
                    "name": "weather_instructions_with_alert",
                    "template": weather_instructions_with_alert,
                }
            ],
            "output_parser": {"type": "default"},
        }
    },
    "tool_configurations": {
        "geocode_city":        weather_agent_config_specialized["tool_configurations"]["geocode_city"],
        "get_current_weather": weather_agent_config_specialized["tool_configurations"]["get_current_weather"],
        "send_weather_alert":  send_weather_alert_tool,
    },
}

weather_agent_key = delete_and_create_agent(weather_agent_config_with_alert, WEATHER_AGENT_NAME)

Deleted existing agent 'Weather Assistant' (agt_weather_assistant_7065)
Created agent 'Weather Assistant' (key: agt_weather_assistant_b58d)


In [13]:
session_name = f"Weather Demo Alert {datetime.now().strftime('%Y%m%d-%H%M%S')}"
response = requests.post(
    f"{BASE_URL}/agents/{weather_agent_key}/sessions",
    headers=headers,
    json={"name": session_name, "metadata": {"purpose": "web_get_demo_post"}},
)
response.raise_for_status()
session_key = response.json()["key"]
print(f"Session Created: {session_key}")

question = "What's the weather in Anchorage right now? Send me an alert."
print(f"\nUser: {question}")
print("=" * 80)

answer = ask_weather(weather_agent_key, session_key, question, show_events=True, body_preview_chars=400)
print(f"\nAgent: {answer}")

Session Created: ase_weather_demo_alert_20260506-100633_4819

User: What's the weather in Anchorage right now? Send me an alert.

------ Agent Events ------
[geocode_city] GET https://geocoding-api.open-meteo.com/v1/search?name=Anchorage&count=1&language=en&format=json
  headers: {'User-Agent': 'vectara-tutorial/1.0'}
  -> [geocode_city] HTTP 200, body: {"results":[{"id":5879400,"name":"Anchorage","latitude":61.21806,"longitude":-149.90028,"elevation":31.0,"feature_code":"PPLA2","country_code":"US","admin1_id":5879092,"admin2_id":5879348,"timezone":"America/Anchorage","population":289600,"postcodes":["99501","99502","99503","99504","99507","99508","99509","99510","99511","99513","99514","99515","99516","99517","99518","99519","99520","99521","995...
[get_current_weather] GET https://api.open-meteo.com/v1/forecast?latitude=61.21806&longitude=-149.90028&current=temperature_2m,weather_code,wind_speed_10m&temperature_unit=celsius&wind_speed_unit=kmh
  headers: {'User-Agent': 'vectara-tutor

The trace now shows **three** tool calls:

- `[geocode_city] GET ...` — same as before, gets coordinates.
- `[get_current_weather] GET ...` — same as before, gets temperature/wind/code.
- `[send_weather_alert] POST https://ntfy.sh/<topic>` — *new*. The `body` field is the alert sentence the LLM wrote, and the `headers` line shows the `Title`, `Priority`, and `Tags` it chose this turn — the system prompt and tool description require all three on every call. The `method` and target URL are fixed (we pinned them in `_POST_LOCKDOWN`); everything else is LLM-chosen per turn.

Because we pinned `url` and `method` in `argument_override`, the LLM cannot accidentally `GET` this endpoint or `POST` somewhere else. That's the textbook reason to register the same `web_get` tool multiple times under different names: each registration carries its own guardrails.

In [14]:
# Read the topic back to prove the POST landed. ntfy.sh's /json poll endpoint
# returns one JSON object per line (NDJSON).
import time
time.sleep(1)  # ntfy publishes near-instantly; tiny buffer for ordering

resp = requests.get(
    f"https://ntfy.sh/{ALERT_TOPIC}/json",
    params={"poll": 1, "since": "5m"},
    timeout=10,
)
resp.raise_for_status()

print(f"Messages on topic '{ALERT_TOPIC}':\n")
for line in resp.text.strip().splitlines():
    msg = json.loads(line)
    print(f"  [priority {msg.get('priority', '?')}] {msg.get('title', '(no title)')}")
    print(f"      {msg.get('message', '')}")
    print(f"      tags: {msg.get('tags', [])}\n")

Messages on topic 'vectara-demo-5cbed713':

  [priority ?] (no title)
      It's currently 6.1°C in Anchorage with light winds at 0.8 km/h and partly cloudy skies.
      tags: []



## Step 7: How the Agent Handles HTTP Errors

Real APIs fail. Endpoints get renamed, services have outages, payloads exceed limits. When `web_get` hits a non-2xx response, the agent has to do something — and the interesting question isn't whether `web_get` *can* return a 4xx/5xx, it's **what the agent does** with one.

In this step we'll:

1. Run a single small demo where the agent hits a real 404 from a real host (a typo'd Open-Meteo path) and see how it responds in the trace.
2. Then enumerate the patterns to use in your own agents to handle failures gracefully — system-prompt language, a reusable failure-detection helper, and the `argument_override` knobs that shape failure behavior.

We deliberately do **not** simulate a 500 from a fake endpoint. The interesting failure-mode demos are the ones that mirror real engineer mistakes (wrong path, deprecated endpoint, typo) — those happen organically against real hosts.

In [15]:
# A *separate* agent with a deliberately wrong path on the real Open-Meteo host.
# Open-Meteo's geocoding endpoint actually lives at /v1/search, not /v1/search-cities,
# so any call this agent makes will return a real 404 from a real server.

BROKEN_AGENT_NAME = "Weather Assistant (broken endpoint)"

weather_agent_config_broken = {
    "name": BROKEN_AGENT_NAME,
    "description": "Weather agent intentionally pointing at a non-existent path to demo 404 handling.",
    "model": {"name": "gpt-4o"},
    "first_step_name": "main",
    "steps": {
        "main": {
            "instructions": [
                {
                    "type": "inline",
                    "name": "weather_instructions_specialized",
                    "template": weather_instructions_specialized,
                }
            ],
            "output_parser": {"type": "default"},
        }
    },
    "tool_configurations": {
        "geocode_city": {
            "type": "web_get",
            "description_template": (
                # /v1/search-cities does NOT exist; the real path is /v1/search.
                # This produces an organic 404 — the kind of mistake a real engineer
                # makes when typing or following stale documentation.
                "Resolve a city name to latitude and longitude using Open-Meteo's "
                "geocoding API. Construct the URL as: "
                "https://geocoding-api.open-meteo.com/v1/search-cities"
                "?name=<CITY>&count=1&language=en&format=json. "
                "From the JSON response, read results[0].latitude and "
                "results[0].longitude."
            ),
            "argument_override": _LOCKDOWN,
        },
        # The forecast tool is unchanged from Step 5; it should never be reached
        # in this demo because the geocode call fails first.
        "get_current_weather": (
            weather_agent_config_specialized["tool_configurations"]["get_current_weather"]
        ),
    },
}

broken_agent_key = delete_and_create_agent(weather_agent_config_broken, BROKEN_AGENT_NAME)

Created agent 'Weather Assistant (broken endpoint)' (key: agt_weather_assistant_broken_endpoint_a5fd)


In [16]:
session_name = f"Weather Demo Broken {datetime.now().strftime('%Y%m%d-%H%M%S')}"
response = requests.post(
    f"{BASE_URL}/agents/{broken_agent_key}/sessions",
    headers=headers,
    json={"name": session_name, "metadata": {"purpose": "web_get_demo_404"}},
)
response.raise_for_status()
broken_session_key = response.json()["key"]
print(f"Session Created: {broken_session_key}")

question = "What's the weather in Madrid right now?"
print(f"\nUser: {question}")
print("=" * 80)

# Capture both the printed trace AND the raw events list, so we can hand
# the events to `find_tool_failures` in the next cell.
answer, broken_events = ask_weather(
    broken_agent_key, broken_session_key, question,
    show_events=True, return_events=True,
)
print(f"\nAgent: {answer}")

Session Created: ase_weather_demo_broken_20260506-100645_ed1a

User: What's the weather in Madrid right now?

------ Agent Events ------
[geocode_city] GET https://geocoding-api.open-meteo.com/v1/search-cities?name=Madrid&count=1&language=en&format=json
  headers: {'User-Agent': 'vectara-tutorial/1.0'}
  -> [geocode_city] HTTP 404, body: {"error":true,"reason":"Not Found"}
--------------------------

Agent: I encountered an issue while trying to get the geographical coordinates for Madrid. Please try again later!


The trace shows the failure clearly: `[geocode_city] GET https://geocoding-api.open-meteo.com/v1/search-cities?...` returns `HTTP 404`, and the agent's reply admits the lookup failed instead of inventing a temperature. That's the system prompt's *"if a tool call fails, say so honestly and do not invent values"* doing its job — and that line is what makes the difference between a graceful failure and a hallucination.

### Patterns for handling failures in your own agents

#### a) System-prompt language for graceful failure

Drop wording like this into your `instructions` template:

> *If a `web_get` call returns a non-2xx status code, do not invent values or fabricate data. Tell the user the request failed and identify which endpoint failed. Do not retry the same URL more than once in a single turn.*

The "no retry within a turn" clause matters: without it, an LLM seeing a 404 will sometimes loop on the same broken URL until the turn budget runs out.

In [17]:
def find_tool_failures(events):
    """Return every tool_output event whose status code indicates failure.

    Treats anything outside 200-399 (including missing/None) as a failure.
    Returns a list of (tool_name, status_code, url, body_preview) tuples,
    in event order. Lift this into your own caller for monitoring, alerting,
    or driving caller-side retry logic.
    """
    failures = []
    last_url_per_tool = {}  # match each tool_output to the URL its tool_input used
    for event in events:
        etype = event.get("type")
        tname = event.get("tool_configuration_name", "?")
        if etype == "tool_input":
            last_url_per_tool[tname] = event.get("tool_input", {}).get("url", "")
        elif etype == "tool_output":
            to = event.get("tool_output", {}) or {}
            status = to.get("status_code")
            if status is None or status < 200 or status >= 400:
                body = to.get("content", "")
                if isinstance(body, (dict, list)):
                    body = json.dumps(body)
                failures.append((tname, status, last_url_per_tool.get(tname, ""), str(body)[:200]))
    return failures


# Run it on the events from the broken-endpoint demo above.
failures = find_tool_failures(broken_events)
print(f"Found {len(failures)} tool failure(s):\n")
for tool, status, url, body in failures:
    print(f"  [{tool}] HTTP {status}")
    print(f"     url:  {url}")
    print(f"     body: {body}\n")

Found 1 tool failure(s):

  [geocode_city] HTTP 404
     url:  https://geocoding-api.open-meteo.com/v1/search-cities?name=Madrid&count=1&language=en&format=json
     body: {"error":true,"reason":"Not Found"}



#### b) Caller-side failure detection

The cell above defines `find_tool_failures(events)` — a small utility that scans the events Vectara returns and surfaces every `tool_output` whose status is missing or not in the 2xx/3xx range. Use it for monitoring (alert when failure rate spikes), audit logging (record which endpoints failed), or driving caller-side retry.

#### c) `argument_override` knobs that shape failure behavior

Step 4 already introduced these; for failure handling specifically:

- **`timeout_seconds`** — caps how long a single request can take. Sets a hard ceiling on how slow an upstream API can drag your turn budget.
- **`max_content_bytes`** — caps the response body size that gets fed back to the LLM. A 50MB JSON dump can blow your context window; pin this defensively.
- **`ssl_verify`** — keep `True` in production; disabling it is a common lazy fix for cert errors that masks real problems.
- **`follow_redirects`** — leave `True` for normal APIs; set `False` to detect redirect loops or unexpected hostname changes.

Tighter limits don't *prevent* failures; they bound the blast radius when one happens.

#### d) Retry behavior

Whether `web_get` retries internally on transient 5xx is implementation-defined and not part of the public tool spec at the time of writing — don't assume a specific count. The robust pattern is **caller-side**: collect `find_tool_failures(events)` and decide per-call whether the failure is retryable. A 502/503/504 on a `GET` is usually safe to retry with backoff; 4xx and `POST`s usually aren't (POSTs may not be idempotent — re-trying a `send_weather_alert` could deliver duplicate notifications).

To check your environment's actual retry count, point a throwaway agent at `https://httpbin.org/status/503` (always returns 503) and count how many `tool_input` events Vectara emits for that tool per turn — one means no internal retries, more than one means there are.

## Cleanup (Optional)

Delete both agents created in this notebook (the alert agent from Step 6 and the broken-endpoint agent from Step 7) so you don't accumulate test resources.

In [18]:
# Delete both agents created across Steps 6 and 7. `broken_agent_key` only
# exists if Step 7 was executed in this kernel; fall back to None otherwise so
# the cell still works for readers who stopped at Step 6.
to_delete = [
    ("alert agent", weather_agent_key),
    ("broken-endpoint agent", globals().get("broken_agent_key")),
]
for label, key in to_delete:
    if not key:
        continue
    response = requests.delete(f"{BASE_URL}/agents/{key}", headers=headers)
    if response.status_code == 204:
        print(f"Deleted {label}: {key}")
    else:
        print(f"Error deleting {label} ({key}): {response.status_code} - {response.text}")

Deleted alert agent: agt_weather_assistant_b58d
Deleted broken-endpoint agent: agt_weather_assistant_broken_endpoint_a5fd
